# Netflix Content Interest Analysis

**Group members:** Add names here  
**Course:** Add course and section here  
**Video presentation:** Add the final video link here

## Project overview

Netflix needs to decide what content is worth acquiring, producing, or promoting. Our project uses the supplied Netflix catalog data to understand the platform's current content mix. We then use publicly available IMDb title data as an outside measure of audience interest. We test whether basic title characteristics, such as format, genre, release year, and runtime, can help predict how much public attention a movie or TV series receives.

This does not recreate Netflix's private viewing data. Instead, it gives Netflix a simple first-screening tool for comparing titles when internal viewing data is not available.

## Business problem and context

Netflix competes for viewers' limited attention. The company says it evaluates content using expected viewing, engagement, and acquisition impact, while also considering the cost of each licensing deal. This makes early signals of audience interest useful when Netflix is deciding which titles to acquire or promote.

Our question is: **Can publicly available title characteristics help predict audience interest in movies and TV series?** We use the number of IMDb votes as an outside proxy for public attention. A title with more votes has reached more viewers or generated more interest, although votes are not the same thing as Netflix views.

### Sources for business context

1. Netflix Investor Relations. *Top Investor Questions: Content Strategy and Licensing.* https://ir.netflix.net/ir-overview/top-investor-questions/
2. Deloitte. *2025 Digital Media Trends.* https://www.deloitte.com/us/en/insights/industry/technology/digital-media-trends-consumption-habits-survey/2025.html

Netflix is the company source. Deloitte is an independent source about the streaming industry and consumer behavior.

## Data sources

**Netflix catalog dataset:** Use the supplied `netflix_titles.csv` file. It contains 8,807 Netflix titles and includes title type, country, date added, maturity rating, duration, genres, cast, director, and description. It is used for Netflix-specific catalog analysis.

**Primary modeling dataset:** IMDb Non-Commercial Datasets: https://developer.imdb.com/non-commercial-datasets/

We use two IMDb files that can be joined with IMDb's title ID (`tconst`):

- `title.basics.tsv.gz`: title type, release year, runtime, and genres
- `title.ratings.tsv.gz`: average IMDb rating and total number of votes

The final target is `log_votes`, which is the natural log of the number of votes. We use a log because vote counts are very skewed: a small number of titles have much more attention than most titles.

## 1. Import libraries

These are the same main libraries used in class: pandas for data, matplotlib/seaborn for charts, and scikit-learn for modeling.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor

ModuleNotFoundError: No module named 'numpy'

## 2. Load the Netflix catalog data

Unzip one of the supplied files and put `netflix_titles.csv` in a `data` folder next to this notebook. The two supplied files are identical, so only use one copy.

In [ ]:
netflix_catalog = pd.read_csv('data/netflix_titles.csv')

print(netflix_catalog.shape)
display(netflix_catalog.head())
print(netflix_catalog.columns.tolist())

## 3. Clean the Netflix catalog data

We create simpler date, genre, and country fields for the Netflix catalog charts. These steps do not change the original file.

In [ ]:
netflix_catalog['date_added'] = pd.to_datetime(
    netflix_catalog['date_added'].str.strip(), errors='coerce'
)
netflix_catalog['primary_genre'] = netflix_catalog['listed_in'].str.split(',').str[0]
netflix_catalog['primary_country'] = netflix_catalog['country'].str.split(',').str[0]

print(netflix_catalog.isna().sum())

## 4. Netflix catalog charts

These charts describe the Netflix catalog our company is making decisions about.

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=netflix_catalog, x='type', color='#E50914')
plt.title('Netflix Catalog Titles by Type')
plt.xlabel('Title Type')
plt.ylabel('Number of Titles')
plt.show()

In [ ]:
top_catalog_genres = netflix_catalog['primary_genre'].value_counts().head(10)
plt.figure(figsize=(9, 5))
sns.barplot(x=top_catalog_genres.values, y=top_catalog_genres.index, color='#221F1F')
plt.title('Most Common Primary Genres in the Netflix Catalog')
plt.xlabel('Number of Titles')
plt.ylabel('Primary Genre')
plt.show()

In [ ]:
catalog_years = netflix_catalog['release_year'].value_counts().sort_index()
plt.figure(figsize=(10, 4))
plt.plot(catalog_years.index, catalog_years.values, color='#E50914')
plt.title('Netflix Catalog Titles by Release Year')
plt.xlabel('Release Year')
plt.ylabel('Number of Titles')
plt.show()

## 5. Load the IMDb data

The files are downloaded directly from IMDb. The full `title.basics` file is large, so we load only the columns needed for this project.

In [ ]:
basics_url = 'https://datasets.imdbws.com/title.basics.tsv.gz'
ratings_url = 'https://datasets.imdbws.com/title.ratings.tsv.gz'

basics = pd.read_csv(
    basics_url, sep='\t', na_values='\\N',
    usecols=['tconst', 'titleType', 'primaryTitle', 'startYear', 'runtimeMinutes', 'genres']
)
ratings = pd.read_csv(ratings_url, sep='\t')

print(basics.shape)
print(ratings.shape)

## 3. Inspect the raw data

Before cleaning, we check the columns and look at a few rows from each file.

In [ ]:
display(basics.head())
display(ratings.head())

print(basics.info())
print(ratings.info())

## 4. Keep the title types that fit Netflix's content decision

We focus on movies and TV series. We also keep titles released from 2000 through 2025 so that the data reflects more modern viewing behavior.

In [ ]:
content = basics[basics['titleType'].isin(['movie', 'tvSeries'])].copy()
content['startYear'] = pd.to_numeric(content['startYear'], errors='coerce')
content['runtimeMinutes'] = pd.to_numeric(content['runtimeMinutes'], errors='coerce')

content = content[content['startYear'].between(2000, 2025)]
content = content.merge(ratings, on='tconst', how='inner')
print(content.shape)

## 5. Clean the data

A title needs a known year, runtime, genre, and vote count to be useful for this model. We remove duplicate title IDs and limit the data to titles with at least 25 votes so a few random votes do not drive the analysis.

In [ ]:
content = content.drop_duplicates(subset='tconst')
content = content.dropna(subset=['startYear', 'runtimeMinutes', 'genres', 'numVotes'])
content = content[content['numVotes'] >= 25]
content = content[content['runtimeMinutes'].between(20, 300)]

print(content.isna().sum())
print(content.shape)

## 6. Create model-ready features

IMDb stores up to three genres in one text field. We split them into separate columns. We also create the target variable, `log_votes`, so extremely popular titles do not overpower the model.

In [ ]:
genre_parts = content['genres'].str.split(',', expand=True)
content['genre_1'] = genre_parts[0]
content['genre_2'] = genre_parts[1].fillna('None')
content['genre_3'] = genre_parts[2].fillna('None')
content['log_votes'] = np.log1p(content['numVotes'])

model_data = content.dropna(subset=['genre_1']).copy()
print(model_data.shape)

## 7. Make the project manageable

IMDb has more rows than we need for a class project. We take a random sample of 30,000 titles. This is still a large dataset, but it allows the models to run in a reasonable amount of time.

In [ ]:
sample_size = min(30000, len(model_data))
model_data = model_data.sample(n=sample_size, random_state=42)

features = ['titleType', 'startYear', 'runtimeMinutes', 'genre_1', 'genre_2', 'genre_3']
target = 'log_votes'

model_data[features + [target]].head()

# Exploratory data analysis

The following charts help us understand the titles before modeling. These charts can also be refined and used in the final presentation.

## 8. Chart 1: Number of titles by type

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=model_data, x='titleType', color='#E50914')
plt.title('Number of Titles by Type')
plt.xlabel('Title Type')
plt.ylabel('Number of Titles')
plt.show()

## 9. Chart 2: Most common primary genres

In [ ]:
top_genres = model_data['genre_1'].value_counts().head(10)
plt.figure(figsize=(9, 5))
sns.barplot(x=top_genres.values, y=top_genres.index, color='#221F1F')
plt.title('Most Common Primary Genres')
plt.xlabel('Number of Titles')
plt.ylabel('Primary Genre')
plt.show()

## 10. Chart 3: Releases by year

In [ ]:
year_counts = model_data['startYear'].value_counts().sort_index()
plt.figure(figsize=(10, 4))
plt.plot(year_counts.index, year_counts.values, color='#E50914')
plt.title('Titles Released by Year')
plt.xlabel('Release Year')
plt.ylabel('Number of Titles')
plt.show()

## 11. Chart 4: Distribution of the target

The log transformation makes the distribution easier to model and compare.

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(model_data['log_votes'], bins=30, color='#E50914')
plt.title('Distribution of Log IMDb Vote Counts')
plt.xlabel('Log Number of Votes')
plt.ylabel('Number of Titles')
plt.show()

## 12. Chart 5: Average audience interest by primary genre

In [ ]:
genre_interest = model_data.groupby('genre_1')['log_votes'].mean().sort_values(ascending=False).head(10)
plt.figure(figsize=(9, 5))
sns.barplot(x=genre_interest.values, y=genre_interest.index, color='#E50914')
plt.title('Average Audience Interest by Primary Genre')
plt.xlabel('Average Log IMDb Votes')
plt.ylabel('Primary Genre')
plt.show()

## 13. Split the data

We use 80% of the data to train the models and hold back 20% for testing. The test set is not used while the models are learning.

In [ ]:
X = model_data[features]
y = model_data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(X_train.shape)
print(X_test.shape)

## 14. Prepare numeric and categorical columns

The models need numeric values. This preprocessing step scales numeric columns when needed and changes genres and title type into usable indicator columns.

In [ ]:
numeric_features = ['startYear', 'runtimeMinutes']
categorical_features = ['titleType', 'genre_1', 'genre_2', 'genre_3']

linear_prep = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

forest_prep = ColumnTransformer([
    ('num', 'passthrough', numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
])

# Modeling approach

We compare three regression models. Linear Regression gives us a simple baseline. K-Nearest Neighbors looks for similar titles. Random Forest can capture more complex relationships between features and audience interest.

## 15. Model 1: Linear Regression

This baseline model shows how well a simple linear relationship performs.

In [ ]:
linear_model = Pipeline([
    ('prep', linear_prep),
    ('model', LinearRegression())
])

linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)
print('Linear Regression is complete.')

## 16. Model 2: K-Nearest Neighbors Regression

We test a few values of K and choose the setting with the best cross-validation score.

In [ ]:
knn_model = Pipeline([
    ('prep', linear_prep),
    ('model', KNeighborsRegressor())
])

knn_grid = {'model__n_neighbors': [5, 10, 20]}
knn_search = GridSearchCV(knn_model, knn_grid, cv=3, scoring='neg_mean_absolute_error')
knn_search.fit(X_train, y_train)
knn_pred = knn_search.predict(X_test)
print(knn_search.best_params_)

## 17. Model 3: Random Forest Regression

Random Forest creates many decision trees and combines their predictions. We test different tree depths to avoid making the model more complex than necessary.

In [ ]:
forest_model = Pipeline([
    ('prep', forest_prep),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])

forest_grid = {'model__n_estimators': [100], 'model__max_depth': [10, 20, None]}
forest_search = GridSearchCV(forest_model, forest_grid, cv=3, scoring='neg_mean_absolute_error')
forest_search.fit(X_train, y_train)
forest_pred = forest_search.predict(X_test)
print(forest_search.best_params_)

# Model evaluation

We use MAE, RMSE, and R-squared. Lower MAE and RMSE are better. Higher R-squared is better.

In [ ]:
def get_metrics(actual, predicted):
    return {
        'MAE': mean_absolute_error(actual, predicted),
        'RMSE': mean_squared_error(actual, predicted, squared=False),
        'R-squared': r2_score(actual, predicted)
    }

results = pd.DataFrame([
    get_metrics(y_test, linear_pred),
    get_metrics(y_test, knn_pred),
    get_metrics(y_test, forest_pred)
], index=['Linear Regression', 'KNN Regression', 'Random Forest'])
results.sort_values('MAE')

## 18. Chart 6: Compare model error

This chart makes it easy to explain which model made the smallest average error.

In [ ]:
plt.figure(figsize=(8, 4))
results.sort_values('MAE')['MAE'].plot(kind='bar', color='#E50914')
plt.title('Model Comparison: Mean Absolute Error')
plt.xlabel('Model')
plt.ylabel('MAE for Log IMDb Votes')
plt.xticks(rotation=0)
plt.show()

## 19. Actual versus predicted results for the best model

This chart helps us see whether the selected model's predictions generally move with the real outcomes.

In [ ]:
best_name = results['MAE'].idxmin()
best_predictions = {
    'Linear Regression': linear_pred,
    'KNN Regression': knn_pred,
    'Random Forest': forest_pred
}[best_name]

plt.figure(figsize=(6, 6))
plt.scatter(y_test, best_predictions, alpha=0.25, color='#E50914')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='black')
plt.title(f'Actual vs. Predicted: {best_name}')
plt.xlabel('Actual Log IMDb Votes')
plt.ylabel('Predicted Log IMDb Votes')
plt.show()

## 20. Feature importance for Random Forest

Feature importance shows which inputs had the most influence in the Random Forest model. This chart is useful even if Random Forest is not the final best model.

In [ ]:
feature_names = forest_search.best_estimator_.named_steps['prep'].get_feature_names_out()
importance = forest_search.best_estimator_.named_steps['model'].feature_importances_
importance_df = pd.DataFrame({'feature': feature_names, 'importance': importance})
importance_df = importance_df.sort_values('importance', ascending=False).head(10)

plt.figure(figsize=(9, 5))
sns.barplot(data=importance_df, x='importance', y='feature', color='#E50914')
plt.title('Top Random Forest Features')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

# Conclusions and recommendations

## What we found

After running the notebook, replace the brackets below with your team's actual results.

- The best model was **[model name]** because it had the lowest MAE of **[value]**.
- The most useful title characteristics were **[features from the feature-importance chart or model results]**.
- This means basic public title information has **[limited/moderate/strong]** ability to estimate external audience interest.

## Recommendation for Netflix

Netflix should use a model like this as an early screening input when comparing possible licensed titles or planning promotion. It should not be the only decision tool. Netflix should combine the prediction with its own private data on member viewing behavior, content costs, regional demand, and expected retention impact.

## Limitations and next steps

IMDb vote counts are an outside proxy for interest, not Netflix viewing. The model also cannot account for marketing, release timing, platform availability, or Netflix's private member preferences. A stronger next version would merge public Netflix engagement data with title metadata and use the same modeling process to predict Netflix views directly.